# 02 - Hiperparametros, Poda y Control del Sobreajuste en Arboles de Regresion

## Curso de Machine Learning: Modelos de Regresion

En el cuaderno anterior demostramos que un arbol de decision sin restricciones memoriza los datos de entrenamiento ($R^2_{train} = 1.0$) y generaliza pobremente. Este fenomeno se conoce como **sobreajuste (overfitting)** y es el principal desafio practico de los arboles CART.

En este cuaderno estudiaremos sistematicamente los **hiperparametros de regularizacion** que permiten controlar la complejidad del arbol, y la tecnica avanzada de **Poda de Complejidad de Costo (Cost-Complexity Pruning, CCP)** que Breiman (1984) propuso como mecanismo formal para encontrar el arbol optimo en terminos del balance sesgo-varianza.

---

### Contenidos
1. Diagnostico del Sobreajuste: $R^2_{train}$ vs. $R^2_{test}$
2. Hiperparametro `max_depth`: Profundidad Maxima
3. Hiperparametro `min_samples_split`: Muestras Minimas para Dividir
4. Hiperparametro `min_samples_leaf`: Muestras Minimas por Hoja
5. Hiperparametro `max_leaf_nodes`: Numero Maximo de Hojas
6. Curvas de Validacion: Deteccion Sistematica del Sobreajuste
7. Poda de Complejidad de Costo (CCP): Teoria y el Parametro `ccp_alpha`
8. Seleccion del Arbol Optimo con Validacion Cruzada y CCP
9. Comparativa Final: Arbol Podado vs. OLS vs. Ridge
10. Conclusiones


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.tree import DecisionTreeRegressor, plot_tree
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_squared_error, r2_score

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
np.random.seed(42)


## 1. Diagnostico del Sobreajuste

El sobreajuste en arboles de decision es particularmente facil de diagnosticar. Observamos la brecha entre el rendimiento en entrenamiento y en prueba a medida que se incrementa la complejidad del modelo (profundidad del arbol).


In [ ]:
# Carga de datos
diabetes = load_diabetes(scaled=False, as_frame=True)
X = diabetes.frame.drop(columns=['target'])
y = diabetes.frame['target']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

# Evaluacion sistematica: profundidad vs rendimiento
profundidades = range(1, 21)
r2_train_list = []
r2_test_list = []
n_hojas_list = []

for d in profundidades:
    tree = DecisionTreeRegressor(max_depth=d, random_state=42)
    tree.fit(X_train, y_train)
    r2_train_list.append(r2_score(y_train, tree.predict(X_train)))
    r2_test_list.append(r2_score(y_test, tree.predict(X_test)))
    n_hojas_list.append(tree.get_n_leaves())

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Panel 1: R2 en Train vs Test
axes[0].plot(profundidades, r2_train_list, 'o-', color='#1f77b4', linewidth=2, label='$R^2$ Entrenamiento')
axes[0].plot(profundidades, r2_test_list, 's-', color='#d62728', linewidth=2, label='$R^2$ Prueba')
axes[0].set_xlabel('Profundidad Maxima (max_depth)', fontsize=11, fontweight='bold')
axes[0].set_ylabel('Coeficiente de Determinacion ($R^2$)', fontsize=11, fontweight='bold')
axes[0].set_title('Diagnostico de Sobreajuste: $R^2$ vs. Profundidad', fontsize=12, fontweight='bold')
axes[0].legend(fontsize=11)
axes[0].axhline(1.0, color='gray', linestyle=':', alpha=0.5)

# Panel 2: Numero de Hojas vs Profundidad
axes[1].plot(profundidades, n_hojas_list, 'o-', color='#2ca02c', linewidth=2)
axes[1].set_xlabel('Profundidad Maxima (max_depth)', fontsize=11, fontweight='bold')
axes[1].set_ylabel('Numero de Hojas Terminales', fontsize=11, fontweight='bold')
axes[1].set_title('Crecimiento Exponencial de la Complejidad del Arbol', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

# Identificar la profundidad optima
mejor_d = profundidades[np.argmax(r2_test_list)]
print(f"Profundidad con mejor R2 en Test: max_depth={mejor_d} (R2_test={max(r2_test_list):.4f})")


## 2. Hiperparametro `max_depth`: Profundidad Maxima

`max_depth` limita la distancia maxima entre el nodo raiz y cualquier hoja terminal. Es el control mas intuitivo y frecuentemente el mas efectivo.

- **Valores bajos** (ej. 2-4): Arbol muy simple (alto sesgo, baja varianza). Funciona como modelo de reglas de negocio.
- **Valores altos** (ej. 10-20): Arbol complejo (bajo sesgo, alta varianza). Riesgo de memorizar ruido.
- **`None` (sin limite)**: El arbol crece hasta que cada hoja contiene una unica observacion (overfitting maximo).

Ya hemos visualizado este efecto en la celda anterior. A continuacion analizaremos los demas hiperparametros.


## 3. Hiperparametros `min_samples_split` y `min_samples_leaf`

### `min_samples_split` (Valor por defecto: 2)
Numero minimo de muestras requeridas en un nodo para que este sea candidato a dividirse. Si un nodo contiene menos muestras que este umbral, se convierte automaticamente en hoja terminal.

### `min_samples_leaf` (Valor por defecto: 1)
Numero minimo de muestras que debe contener **cada** hoja terminal resultante de una division. Si una division potencial produce una hoja con menos muestras que este umbral, esa division se descarta.

Ambos parametros previenen que el arbol cree regiones con muy pocas observaciones, donde la media estimada es estadisticamente inestable y tiene altisima varianza.


In [ ]:
# Efecto de min_samples_leaf sobre el sobreajuste
min_leaf_values = [1, 5, 10, 20, 30, 50]
resultados_leaf = []

for ml in min_leaf_values:
    tree_ml = DecisionTreeRegressor(min_samples_leaf=ml, random_state=42)
    tree_ml.fit(X_train, y_train)
    
    resultados_leaf.append({
        'min_samples_leaf': ml,
        'Hojas': tree_ml.get_n_leaves(),
        'Profundidad': tree_ml.get_depth(),
        'R2_Train': r2_score(y_train, tree_ml.predict(X_train)),
        'R2_Test': r2_score(y_test, tree_ml.predict(X_test)),
        'MSE_Test': mean_squared_error(y_test, tree_ml.predict(X_test))
    })

df_leaf = pd.DataFrame(resultados_leaf)
print("Efecto de min_samples_leaf en la Complejidad y Rendimiento:")
print(df_leaf.round(4).to_string(index=False))


## 4. Hiperparametro `max_leaf_nodes`

`max_leaf_nodes` establece un limite superior absoluto al numero de hojas terminales del arbol. El algoritmo de construccion selecciona en cada iteracion la hoja cuya division produce la mayor reduccion de impureza global, hasta alcanzar el limite especificado. Esto produce arboles con una topologia potencialmente desbalanceada pero optimizada globalmente.


In [ ]:
# Efecto de max_leaf_nodes
max_leaves_values = [4, 8, 16, 32, 64, 128]
resultados_leaves = []

for ml in max_leaves_values:
    tree_mln = DecisionTreeRegressor(max_leaf_nodes=ml, random_state=42)
    tree_mln.fit(X_train, y_train)
    
    resultados_leaves.append({
        'max_leaf_nodes': ml,
        'Profundidad': tree_mln.get_depth(),
        'R2_Train': r2_score(y_train, tree_mln.predict(X_train)),
        'R2_Test': r2_score(y_test, tree_mln.predict(X_test)),
        'MSE_Test': mean_squared_error(y_test, tree_mln.predict(X_test))
    })

df_leaves = pd.DataFrame(resultados_leaves)
print("Efecto de max_leaf_nodes en la Complejidad y Rendimiento:")
print(df_leaves.round(4).to_string(index=False))


## 5. Curvas de Validacion con Validacion Cruzada

Para una seleccion robusta del hiperparametro, utilizaremos validacion cruzada de 5 particiones estratificadas para evaluar el rendimiento esperado fuera de muestra a lo largo de una malla de valores de `max_depth`.


In [ ]:
# Curva de validacion con cross_val_score
from sklearn.model_selection import cross_val_score

profundidades_cv = range(1, 16)
cv_scores_mean = []
cv_scores_std = []
train_scores_cv = []

for d in profundidades_cv:
    tree_cv = DecisionTreeRegressor(max_depth=d, random_state=42)
    scores = cross_val_score(tree_cv, X_train, y_train, cv=5, scoring='r2')
    cv_scores_mean.append(scores.mean())
    cv_scores_std.append(scores.std())
    
    tree_cv.fit(X_train, y_train)
    train_scores_cv.append(r2_score(y_train, tree_cv.predict(X_train)))

cv_scores_mean = np.array(cv_scores_mean)
cv_scores_std = np.array(cv_scores_std)

plt.figure(figsize=(10, 5.5))
plt.plot(profundidades_cv, train_scores_cv, 'o-', color='#1f77b4', linewidth=2, label='$R^2$ Entrenamiento')
plt.plot(profundidades_cv, cv_scores_mean, 's-', color='#d62728', linewidth=2, label='$R^2$ CV (media +/- 1 std)')
plt.fill_between(profundidades_cv, cv_scores_mean - cv_scores_std, cv_scores_mean + cv_scores_std,
                 color='#d62728', alpha=0.15)
plt.xlabel('Profundidad Maxima (max_depth)', fontsize=11, fontweight='bold')
plt.ylabel('$R^2$', fontsize=11, fontweight='bold')
plt.title('Curva de Validacion: Seleccion de max_depth por Validacion Cruzada (k=5)', fontsize=12, fontweight='bold')
plt.legend(fontsize=11)
plt.tight_layout()
plt.show()

mejor_d_cv = list(profundidades_cv)[np.argmax(cv_scores_mean)]
print(f"Mejor max_depth por CV: {mejor_d_cv} (R2_CV_media = {max(cv_scores_mean):.4f})")


## 6. Poda de Complejidad de Costo (Cost-Complexity Pruning, CCP)

La **Poda de Complejidad de Costo** es la tecnica mas rigurosa para encontrar el arbol optimo. En lugar de restringir el crecimiento del arbol de manera preventiva (pre-pruning), esta tecnica permite que el arbol crezca completamente y luego lo recorta de abajo hacia arriba eliminando las ramas que aportan la menor reduccion neta de impureza relativa a su costo en complejidad.

### Formulacion Matematica

Para un arbol $T$ con $|T|$ hojas terminales, definimos la **funcion de costo total regularizada**:

$$R_{\alpha}(T) = R(T) + \alpha \cdot |T|$$

Donde:
- $R(T) = \frac{1}{n} \sum_{m=1}^{|T|} n_m \cdot \text{MSE}_m$ es el error total de resubstitucion del arbol (MSE ponderado por el numero de muestras en cada hoja).
- $|T|$ es el numero de hojas terminales (medida de la complejidad del arbol).
- $\alpha \geq 0$ es el **parametro de penalizacion de complejidad** (Cost-Complexity Parameter, `ccp_alpha` en Scikit-Learn).

### Interpretacion del Parametro $\alpha$:
- $\alpha = 0$: No hay penalizacion. El arbol optimo es el arbol completamente crecido ($R^2_{train} = 1.0$).
- $\alpha$ pequeno: Penalizacion debil. Se permite un arbol complejo.
- $\alpha$ grande: Penalizacion fuerte. Se fuerza un arbol simple (pocas hojas). En el limite, el arbol colapsa a una unica hoja (prediccion constante $\bar{y}$).

### Procedimiento de Poda:
Scikit-Learn calcula automaticamente la secuencia de valores criticos de $\alpha$ (denominados `ccp_alphas`) y los arboles podados correspondientes mediante el metodo `.cost_complexity_pruning_path()`.


In [ ]:
# Calculo de la ruta de poda CCP
tree_max = DecisionTreeRegressor(random_state=42)
tree_max.fit(X_train, y_train)

path = tree_max.cost_complexity_pruning_path(X_train, y_train)
ccp_alphas = path.ccp_alphas
impurities = path.impurities

print(f"Numero de arboles en la secuencia de poda: {len(ccp_alphas)}")
print(f"Rango de ccp_alpha: [{ccp_alphas.min():.6f}, {ccp_alphas.max():.2f}]")

# Grafico de impureza vs alpha
plt.figure(figsize=(10, 5))
plt.plot(ccp_alphas[:-1], impurities[:-1], 'o-', markersize=4, color='#2b5c8f', linewidth=1.5)
plt.xlabel('Parametro de Complejidad (ccp_alpha)', fontsize=11, fontweight='bold')
plt.ylabel('Impureza Total del Arbol (MSE Ponderado)', fontsize=11, fontweight='bold')
plt.title('Ruta de Poda CCP: Impureza vs. Penalizacion de Complejidad', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()


## 7. Seleccion del Arbol Optimo con Validacion Cruzada y CCP

Entrenamos un arbol para cada valor de `ccp_alpha` en la secuencia de poda y evaluamos su rendimiento mediante validacion cruzada de 5 particiones.


In [ ]:
# Evaluacion por validacion cruzada para cada alpha
# Limitamos a un subconjunto representativo de alphas para eficiencia
n_alphas_subset = min(80, len(ccp_alphas))
indices_subset = np.linspace(0, len(ccp_alphas) - 1, n_alphas_subset, dtype=int)
alphas_subset = ccp_alphas[indices_subset]

cv_means_ccp = []
cv_stds_ccp = []
train_scores_ccp = []
n_leaves_ccp = []

for alpha in alphas_subset:
    tree_ccp = DecisionTreeRegressor(ccp_alpha=alpha, random_state=42)
    scores = cross_val_score(tree_ccp, X_train, y_train, cv=5, scoring='r2')
    cv_means_ccp.append(scores.mean())
    cv_stds_ccp.append(scores.std())
    
    tree_ccp.fit(X_train, y_train)
    train_scores_ccp.append(r2_score(y_train, tree_ccp.predict(X_train)))
    n_leaves_ccp.append(tree_ccp.get_n_leaves())

cv_means_ccp = np.array(cv_means_ccp)
cv_stds_ccp = np.array(cv_stds_ccp)

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

# Panel 1: R2 vs alpha
axes[0].plot(alphas_subset, train_scores_ccp, 'o-', color='#1f77b4', markersize=4, linewidth=1.5, label='$R^2$ Entrenamiento')
axes[0].plot(alphas_subset, cv_means_ccp, 's-', color='#d62728', markersize=4, linewidth=1.5, label='$R^2$ CV (media)')
axes[0].fill_between(alphas_subset, cv_means_ccp - cv_stds_ccp, cv_means_ccp + cv_stds_ccp,
                     color='#d62728', alpha=0.15)
axes[0].set_xlabel('ccp_alpha', fontsize=11, fontweight='bold')
axes[0].set_ylabel('$R^2$', fontsize=11, fontweight='bold')
axes[0].set_title('Seleccion de ccp_alpha por Validacion Cruzada', fontsize=12, fontweight='bold')
axes[0].legend(fontsize=10)

# Panel 2: Hojas vs alpha
axes[1].plot(alphas_subset, n_leaves_ccp, 'o-', color='#2ca02c', markersize=4, linewidth=1.5)
axes[1].set_xlabel('ccp_alpha', fontsize=11, fontweight='bold')
axes[1].set_ylabel('Numero de Hojas', fontsize=11, fontweight='bold')
axes[1].set_title('Complejidad del Arbol segun ccp_alpha', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

# Mejor alpha
best_idx = np.argmax(cv_means_ccp)
best_alpha = alphas_subset[best_idx]
print(f"Mejor ccp_alpha por CV: {best_alpha:.6f} (R2_CV = {cv_means_ccp[best_idx]:.4f}, Hojas = {n_leaves_ccp[best_idx]})")


## 8. Arbol Podado Optimo y Evaluacion Final


In [ ]:
# Arbol optimo podado
tree_podado = DecisionTreeRegressor(ccp_alpha=best_alpha, random_state=42)
tree_podado.fit(X_train, y_train)
y_pred_podado = tree_podado.predict(X_test)

print(f"Arbol Podado Optimo (ccp_alpha={best_alpha:.6f}):")
print(f"  Profundidad:    {tree_podado.get_depth()}")
print(f"  Hojas:          {tree_podado.get_n_leaves()}")
print(f"  MSE en Test:    {mean_squared_error(y_test, y_pred_podado):.2f}")
print(f"  R2 en Test:     {r2_score(y_test, y_pred_podado):.4f}")

# Visualizacion si el arbol es interpretable (pocas hojas)
if tree_podado.get_n_leaves() <= 20:
    plt.figure(figsize=(20, 10))
    plot_tree(tree_podado, feature_names=X.columns.tolist(), filled=True, rounded=True,
              fontsize=8, proportion=True, precision=1)
    plt.title(f'Arbol Podado Optimo (ccp_alpha={best_alpha:.6f}, Hojas={tree_podado.get_n_leaves()})',
              fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()


## 9. Comparativa Final: Arbol Podado vs. Modelos Lineales


In [ ]:
# Modelos de referencia
from sklearn.linear_model import RidgeCV

modelo_ols = LinearRegression().fit(X_train, y_train)
modelo_ridge = RidgeCV(cv=5).fit(X_train, y_train)
tree_full = DecisionTreeRegressor(random_state=42).fit(X_train, y_train)
tree_d3 = DecisionTreeRegressor(max_depth=3, random_state=42).fit(X_train, y_train)

modelos_finales = {
    'OLS': modelo_ols,
    'Ridge (CV)': modelo_ridge,
    'Arbol (max_depth=3)': tree_d3,
    f'Arbol Podado (ccp={best_alpha:.4f})': tree_podado,
    'Arbol sin Restriccion': tree_full
}

tabla_final = []
for nombre, modelo in modelos_finales.items():
    y_p = modelo.predict(X_test)
    tabla_final.append({
        'Modelo': nombre,
        'MSE_Test': mean_squared_error(y_test, y_p),
        'R2_Train': r2_score(y_train, modelo.predict(X_train)),
        'R2_Test': r2_score(y_test, y_p),
        'Complejidad': (getattr(modelo, 'get_n_leaves', lambda: 'N/A')() 
                        if hasattr(modelo, 'get_n_leaves') else f"{len(modelo.coef_)+1} coefs")
    })

df_final = pd.DataFrame(tabla_final)
print("Comparativa Final de Modelos de Regresion:")
print(df_final.round(4).to_string(index=False))


## 10. Conclusiones

### Puntos Clave:
1. **El sobreajuste es el enemigo principal** de los arboles de regresion. Sin restricciones, memorizan los datos y generalizan pobremente.
2. **`max_depth`** es el control mas directo e intuitivo. Valores entre 3 y 7 son tipicos para datasets tabulares de tamano moderado.
3. **`min_samples_leaf`** proporciona un control estadistico: exige que cada prediccion este respaldada por un minimo de observaciones, reduciendo la varianza en regiones escasas.
4. **Poda CCP (`ccp_alpha`)** es el metodo mas riguroso. Crece el arbol completo y luego lo recorta de abajo hacia arriba, encontrando el balance optimo entre ajuste y complejidad mediante validacion cruzada.
5. **Los arboles individuales raramente superan a modelos lineales regularizados** en problemas con relaciones aproximadamente lineales (como en el dataset de diabetes). Su verdadero poder se despliega como bloques constructivos de metodos de ensamble (Random Forest, Gradient Boosting), que estudiaremos en modulos posteriores.

En el siguiente cuaderno, **03_Interpretacion_Visual_e_Importancia_Variables.ipynb**, exploraremos las herramientas de interpretacion que hacen de los arboles de decision uno de los modelos mas transparentes y explicables en la practica profesional.
